In [1]:
# Setup
import os
from pathlib import Path
import sys


act_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if act_root not in sys.path:
    sys.path.insert(0, act_root)

ACT_ROOT = Path(act_root)

In [ ]:
# Run ACT TorchVision pipeline
from act.front_end.torchvision_loader.create_specs import TorchVisionSpecCreator


spec_creator = TorchVisionSpecCreator()

# Assuming we've put `simple_cnn.pt` generated from above in the right place,
# the spec creator should load those pre-existing weights in
specs = spec_creator.create_specs_for_data_model_pairs(
    dataset_names=["MNIST"],
    model_names=["simple_cnn"],
)


LOADING: MNIST + simple_cnn
[1/3] Loading dataset (test split)...
  ✓ Loaded 10000 samples
[2/3] Loading model architecture...
  ✓ Loaded custom model from simple_cnn.py
Weights file for model detected, loading weights...
  ✓ Loaded weights for model simple_cnn.py
[3/3] Summary...
  Dataset: 10000 samples (test split)
  Model: 25,710,922 parameters (25,710,922 trainable)
  Batch size: 1
  Preprocessing: Yes

✓ LOADED SUCCESSFULLY


In [3]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

from act.front_end.torchvision_loader.model_definitions import SimpleCNN
from act.util.device_manager import get_default_device


WEIGHTS_PATH = ACT_ROOT / "data/torchvision/MNIST/weights/simple_cnn.pt"

# Transform from `data_model_mapping`:
# {
#     "grayscale_to_rgb": True,  # Repeat channels: (1,28,28) → (3,28,28)
#     "resize_to": (224, 224),    # Then resize to model input size
#     "normalize": {"mean": [0.1307]*3, "std": [0.3081]*3}
# }
mnist_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
    transforms.Normalize(mean=[0.1307] * 3, std=[0.3081] * 3),
])

# Load dataset
test_dataset = MNIST(
    root=ACT_ROOT / "data/torchvision/MNIST/raw",
    transform=mnist_transform,
    train=False
)
num_classes = len(test_dataset.classes)

test_loader = DataLoader(dataset=test_dataset, batch_size=16)

# Load both models
base_model = SimpleCNN(num_classes=num_classes)

with open(WEIGHTS_PATH, "rb") as weights_file:
    state_dict = torch.load(weights_file, map_location=get_default_device())
    base_model.load_state_dict(state_dict)

# All models generated from spec should be the same
act_model = specs[0][2]

# Compare output of both models
for index, data in enumerate(test_loader):
    images, labels = data

    base_outputs = base_model(images)
    act_outputs = act_model(images)

    print(base_outputs)
    print(act_outputs)

IndexError: list index out of range